In [91]:
# Imports
# Imports for file read/write/saving
from AFMReader.ibw import load_ibw
from pathlib import Path
import numpy as np
import tkinter as tk
from tkinter import filedialog
import os
import pandas as pd

# Imports for fitting etc.

from scipy.optimize import curve_fit
import matplotlib.pyplot as plt
import matplotlib.patches as patches

In [92]:
def save_plot_to_folder(fig, folder_path, filename='plot.jpeg'):
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)  # Create the folder if it doesn't exist
    
    full_path = os.path.join(folder_path, filename)
    fig.tight_layout()
    fig.savefig(full_path, dpi=600, bbox_inches='tight')
    print(f"Plot saved to: {full_path}")

In [93]:
def line_level_0th_order(image_array):
    # Calculate the median of each row
    # (Median is usually better than mean for AFM so tall features don't skew the result)
    row_medians = np.median(image_array, axis=1, keepdims=True)
    
    # Subtract the row median from each pixel in that row
    leveled_image = image_array - row_medians
    return leveled_image

In [94]:
def line_level_1st_order(image_array):
    leveled_image = np.zeros_like(image_array)
    x_pixels = np.arange(image_array.shape[1])
    
    # Loop through each row
    for i, row in enumerate(image_array):
        # Fit a 1st-degree polynomial (a straight line: y = mx + c)
        slope, intercept = np.polyfit(x_pixels, row, 1)
        
        # Calculate the trend line
        trend_line = slope * x_pixels + intercept
        
        # Subtract the trend from the raw data
        leveled_image[i] = row - trend_line
        
    return leveled_image

In [95]:
def plane_level(image_array):
    # Get the X and Y coordinates for every pixel in the image
    Y_coords, X_coords = np.indices(image_array.shape)
    
    # Flatten the 2D arrays into 1D lists so we can run the math
    X_flat = X_coords.flatten()
    Y_flat = Y_coords.flatten()
    Z_flat = image_array.flatten()
    
    # Create the design matrix for the plane equation
    A = np.c_[X_flat, Y_flat, np.ones_like(X_flat)]
    
    # Solve for the best-fit plane parameters [a, b, c]
    # rcond=None suppresses a numpy warning
    C, _, _, _ = np.linalg.lstsq(A, Z_flat, rcond=None)
    
    # Reconstruct the 2D background plane using the solved parameters
    background_plane = (C[0] * X_coords) + (C[1] * Y_coords) + C[2]
    
    # Subtract the plane from the original image
    leveled_image = image_array - background_plane
    return leveled_image

In [96]:
def full_leveling(image_data):
    plane_level = plane_level(image_data)
    full_level = line_level_0th_order(plane_level)

    return full_level

In [97]:
def readandsave(filepath, savepath, channel_name=''):     
    if filepath:  # Check if the user selected a file (didn't click Cancel)
        name = Path(filepath).stem
        save_name = f"{savepath}/{name}.npy"
    else: 
         breakpoint

    image_data, scale = load_ibw(file_path=filepath, channel=channel_name)

    if channel_name == "HeightRetrace":
        full_level = full_leveling(image_data)
        np.save(save_name, full_level)
        print(f"Success! Saved leveled array of shape {full_level.shape} as a .npy file.")
        return full_level, scale
    else:
        np.save(save_name, image_data)
        print(f"Success! Saved raw array of shape {image_data.shape} as a .npy file.")
        return image_data, scale



In [98]:
def show_image(image_data, scale, channel_name, save_path, file_path, label = ''):

    # Configure saving
    if file_path:  # Check if the user selected a file (didn't click Cancel)
        name = Path(file_path).stem
    else: 
        breakpoint

    # Scale adjustment 
    pixels_x, pixels_y = image_data.shape
    x_nm = pixels_x*scale
    y_nm = pixels_y *scale
    fig, ax = plt.subplots(figsize=(8,6))
    im = ax.imshow(
        image_data, 
        cmap = 'magma', 
        origin = 'lower', 
        extent = [0,x_nm, 0, y_nm]
        )

    plt.imshow(image_data, cmap='magma', origin='lower')
    #plt.imshow(image_data, cmap='gray', origin='lower')
    
    # Formatting and colormaps
    ax.set_xticks ([])
    ax.set_yticks([])

    scalebar_length = 50
    x_pos = x_nm -scalebar_length - (0.5*x_nm)
    y_pos = 0.05*y_nm
    bar_thickness = y_nm*0.005
    scale_bar = patches.Rectangle(
        (x_pos, y_pos), 
        scalebar_length, 
        bar_thickness, 
        color = 'white', 
        zorder = 5
    )

    ax.add_patch(scale_bar)
    units = r"$micron$"
    ax.text(
        x_pos + (scalebar_length/2), 
        y_pos + bar_thickness + (0.01*y_nm), 
        fr"{scalebar_length/1000}$\mu$m", 
        color = 'white', 
        ha = 'center', 
        va = 'bottom', 
        fontsize = 12, 
        fontweight = 'bold'
    )

    
    plt.colorbar(im, label = f"{label}")
    plt.title(f'AFM Scan: {channel_name}')
    #plt.xlabel('nm')
    #plt.ylabel('nm')
    save_plot_to_folder(fig, save_path, filename=name)
    # Display the plot
    plt.show()
    return

In [103]:
folder_path = filedialog.askdirectory(title="Select raw data folder")
save_path = filedialog.askdirectory(title="Select saving folder for .npy files")
channel_name = f'NapPotentialRetrace'

In [106]:

ibw_files = sorted(list(Path(folder_path).glob("*.ibw")))
for file_path in ibw_files: 
    image_data,scale= readandsave( str(file_path),
                                        save_path, 
                                        channel_name)
#show_image(image_data, scale, channel_name)



21:55:04 | INFO |ibw.py:ibw:load_ibw:73 | Loading image from : X:\ramadan_group\Group Members\Rehmat Goodwin\BA2PbI4 Paper\Testingnpy\D18_RG0115\LightOn\X250903_RG0115_0002_DARK.ibw
21:55:05 | INFO |ibw.py:ibw:load_ibw:82 | [X250903_RG0115_0002_DARK] : Loaded image from : X:\ramadan_group\Group Members\Rehmat Goodwin\BA2PbI4 Paper\Testingnpy\D18_RG0115\LightOn\X250903_RG0115_0002_DARK.ibw
21:55:05 | INFO |ibw.py:ibw:load_ibw:91 | [X250903_RG0115_0002_DARK] : Extracted channel NapPotentialRetrace
21:55:05 | INFO |ibw.py:ibw:load_ibw:102 | [X250903_RG0115_0002_DARK] : Extracted image.
21:55:05 | INFO |ibw.py:ibw:load_ibw:73 | Loading image from : X:\ramadan_group\Group Members\Rehmat Goodwin\BA2PbI4 Paper\Testingnpy\D18_RG0115\LightOn\X250903_RG0115_0003.ibw


Success! Saved raw array of shape (32, 64) as a .npy file.


21:55:05 | INFO |ibw.py:ibw:load_ibw:82 | [X250903_RG0115_0003] : Loaded image from : X:\ramadan_group\Group Members\Rehmat Goodwin\BA2PbI4 Paper\Testingnpy\D18_RG0115\LightOn\X250903_RG0115_0003.ibw
21:55:05 | INFO |ibw.py:ibw:load_ibw:91 | [X250903_RG0115_0003] : Extracted channel NapPotentialRetrace
21:55:05 | INFO |ibw.py:ibw:load_ibw:102 | [X250903_RG0115_0003] : Extracted image.
21:55:05 | INFO |ibw.py:ibw:load_ibw:73 | Loading image from : X:\ramadan_group\Group Members\Rehmat Goodwin\BA2PbI4 Paper\Testingnpy\D18_RG0115\LightOn\X250903_RG0115_0004.ibw


Success! Saved raw array of shape (32, 64) as a .npy file.


21:55:06 | INFO |ibw.py:ibw:load_ibw:82 | [X250903_RG0115_0004] : Loaded image from : X:\ramadan_group\Group Members\Rehmat Goodwin\BA2PbI4 Paper\Testingnpy\D18_RG0115\LightOn\X250903_RG0115_0004.ibw
21:55:06 | INFO |ibw.py:ibw:load_ibw:91 | [X250903_RG0115_0004] : Extracted channel NapPotentialRetrace
21:55:06 | INFO |ibw.py:ibw:load_ibw:102 | [X250903_RG0115_0004] : Extracted image.
21:55:06 | INFO |ibw.py:ibw:load_ibw:73 | Loading image from : X:\ramadan_group\Group Members\Rehmat Goodwin\BA2PbI4 Paper\Testingnpy\D18_RG0115\LightOn\X250903_RG0115_0005.ibw


...

Success! Saved raw array of shape (32, 64) as a .npy file.


21:56:21 | INFO |ibw.py:ibw:load_ibw:82 | [X250903_RG0115_0101] : Loaded image from : X:\ramadan_group\Group Members\Rehmat Goodwin\BA2PbI4 Paper\Testingnpy\D18_RG0115\LightOn\X250903_RG0115_0101.ibw
21:56:21 | INFO |ibw.py:ibw:load_ibw:91 | [X250903_RG0115_0101] : Extracted channel NapPotentialRetrace
21:56:21 | INFO |ibw.py:ibw:load_ibw:102 | [X250903_RG0115_0101] : Extracted image.
21:56:21 | INFO |ibw.py:ibw:load_ibw:73 | Loading image from : X:\ramadan_group\Group Members\Rehmat Goodwin\BA2PbI4 Paper\Testingnpy\D18_RG0115\LightOn\X250903_RG0115_0102.ibw


Success! Saved raw array of shape (32, 64) as a .npy file.


21:56:21 | INFO |ibw.py:ibw:load_ibw:82 | [X250903_RG0115_0102] : Loaded image from : X:\ramadan_group\Group Members\Rehmat Goodwin\BA2PbI4 Paper\Testingnpy\D18_RG0115\LightOn\X250903_RG0115_0102.ibw
21:56:21 | INFO |ibw.py:ibw:load_ibw:91 | [X250903_RG0115_0102] : Extracted channel NapPotentialRetrace
21:56:21 | INFO |ibw.py:ibw:load_ibw:102 | [X250903_RG0115_0102] : Extracted image.


Success! Saved raw array of shape (32, 64) as a .npy file.


In [101]:
'''# 1. Load the height data
# 2. Apply Plane Level to fix overall sample tilt

image_data, scale = load_ibw(file_path=file_path, channel=channel_name)
plane_corrected = plane_level(image_data)

# 3. Apply Line-by-Line Level to fix horizontal scanning artifacts
fully_leveled = line_level_0th_order(plane_corrected)

pixels_x, pixels_y = image_data.shape

x_nm = pixels_x*scale
y_nm = pixels_y *scale


# 4. Save your processed array
#$np.save("leveled_height_data.npy", fully_leveled)

# Plot the 2D image data
# You can change the 'cmap' (colormap) to something like 'magma' or 'gray' if preferred
fig, ax = plt.subplots(figsize=(8,6))
im = ax.imshow(
    image_data, 
    cmap = 'gray', 
    origin = 'lower', 
    extent = [0,x_nm, 0, y_nm]
)
ax.set_xticks ([])
ax.set_yticks([])
plt.colorbar(im, label = "Height(nm)")
scalebar_length = 500
x_pos = x_nm -scalebar_length - (0.5*x_nm)
y_pos = 0.05*y_nm
bar_thickness = y_nm*0.005

scale_bar = patches.Rectangle(
    (x_pos, y_pos), 
    scalebar_length, 
    bar_thickness, 
    color = 'white', 
    zorder = 5
)

ax.add_patch(scale_bar)
units = r"$micron$"
ax.text(
    x_pos + (scalebar_length/2), 
    y_pos + bar_thickness + (0.01*y_nm), 
    fr"{scalebar_length}$\mu$m", 
    color = 'white', 
    ha = 'center', 
    va = 'bottom', 
    fontsize = 12, 
    fontweight = 'bold'
)

plt.title(f"Height {channel_name}")
plt.show()

plt.imshow(fully_leveled, cmap='gray', origin='lower')
# Add a colorbar and labels
plt.colorbar(label='Height nm')
plt.title(f'AFM Scan: {channel_name}')
plt.xlabel('Pixels')
plt.ylabel('Pixels')

plt.show()'''


<>:1: SyntaxWarning: invalid escape sequence '\m'
<>:1: SyntaxWarning: invalid escape sequence '\m'
C:\Users\rehma\AppData\Local\Temp\ipykernel_8252\59185824.py:1: SyntaxWarning: invalid escape sequence '\m'
  '''# 1. Load the height data


'# 1. Load the height data\n# 2. Apply Plane Level to fix overall sample tilt\n\nimage_data, scale = load_ibw(file_path=file_path, channel=channel_name)\nplane_corrected = plane_level(image_data)\n\n# 3. Apply Line-by-Line Level to fix horizontal scanning artifacts\nfully_leveled = line_level_0th_order(plane_corrected)\n\npixels_x, pixels_y = image_data.shape\n\nx_nm = pixels_x*scale\ny_nm = pixels_y *scale\n\n\n# 4. Save your processed array\n#$np.save("leveled_height_data.npy", fully_leveled)\n\n# Plot the 2D image data\n# You can change the \'cmap\' (colormap) to something like \'magma\' or \'gray\' if preferred\nfig, ax = plt.subplots(figsize=(8,6))\nim = ax.imshow(\n    image_data, \n    cmap = \'gray\', \n    origin = \'lower\', \n    extent = [0,x_nm, 0, y_nm]\n)\nax.set_xticks ([])\nax.set_yticks([])\nplt.colorbar(im, label = "Height(nm)")\nscalebar_length = 500\nx_pos = x_nm -scalebar_length - (0.5*x_nm)\ny_pos = 0.05*y_nm\nbar_thickness = y_nm*0.005\n\nscale_bar = patches.Rec

In [102]:
'''# Debugging
# Define your file path and channel
file_path = r"X:\ramadan_group\Group Members\Rehmat Goodwin\BA2PbI4 Paper\Raw Data\AFM\A6\X250722_RG0104_0001.ibw"
channel_name = "HeightRetrace" # Adjust if your specific file uses spaces or different casing

# AFMReader returns the 2D numpy array and the physical scaling factor
image_data, pixel_to_nm_scaling = load_ibw(file_path=file_path, channel=channel_name)

zero_order_image = line_level_0th_order(image_data)
first_order_image = line_level_1st_order(image_data)
plane_leveled = plane_level(image_data)

# Set up the plot
plt.figure(figsize=(8, 6))

# Plot the 2D image data
# You can change the 'cmap' (colormap) to something like 'magma' or 'gray' if preferred
plt.imshow(image_data, cmap='gray', origin='lower')

# Add a colorbar and labels
plt.colorbar(label='Height nm')
plt.title(f'AFM Scan: {channel_name}')
plt.xlabel('Pixels')
plt.ylabel('Pixels')

# Display the plot
plt.show()

# Set up the plot
plt.figure(figsize=(8, 6))

# Plot the 2D image data
# You can change the 'cmap' (colormap) to something like 'magma' or 'gray' if preferred
plt.imshow(zero_order_image, cmap='gray', origin='lower')

# Add a colorbar and labels
plt.colorbar(label='Height nm')
plt.title(f'AFM Scan: {channel_name}')
plt.xlabel('Pixels')
plt.ylabel('Pixels')

# Display the plot
plt.show()

# Plot the 2D image data
# You can change the 'cmap' (colormap) to something like 'magma' or 'gray' if preferred
plt.imshow(first_order_image, cmap='gray', origin='lower')

# Add a colorbar and labels
plt.colorbar(label='Height nm')
plt.title(f'AFM Scan: {channel_name}')
plt.xlabel('Pixels')
plt.ylabel('Pixels')

# Display the plot
plt.show()

# Plot the 2D image data
# You can change the 'cmap' (colormap) to something like 'magma' or 'gray' if preferred
plt.imshow(plane_leveled, cmap='gray', origin='lower')

# Add a colorbar and labels
plt.colorbar(label='Height nm')
plt.title(f'AFM Scan: {channel_name}')
plt.xlabel('Pixels')
plt.ylabel('Pixels')

# Display the plot
plt.show()'''

<>:1: SyntaxWarning: invalid escape sequence '\G'
<>:1: SyntaxWarning: invalid escape sequence '\G'
C:\Users\rehma\AppData\Local\Temp\ipykernel_8252\1471182789.py:1: SyntaxWarning: invalid escape sequence '\G'
  '''# Debugging


'# Debugging\n# Define your file path and channel\nfile_path = r"X:\ramadan_group\\Group Members\\Rehmat Goodwin\\BA2PbI4 Paper\\Raw Data\\AFM\\A6\\X250722_RG0104_0001.ibw"\nchannel_name = "HeightRetrace" # Adjust if your specific file uses spaces or different casing\n\n# AFMReader returns the 2D numpy array and the physical scaling factor\nimage_data, pixel_to_nm_scaling = load_ibw(file_path=file_path, channel=channel_name)\n\nzero_order_image = line_level_0th_order(image_data)\nfirst_order_image = line_level_1st_order(image_data)\nplane_leveled = plane_level(image_data)\n\n# Set up the plot\nplt.figure(figsize=(8, 6))\n\n# Plot the 2D image data\n# You can change the \'cmap\' (colormap) to something like \'magma\' or \'gray\' if preferred\nplt.imshow(image_data, cmap=\'gray\', origin=\'lower\')\n\n# Add a colorbar and labels\nplt.colorbar(label=\'Height nm\')\nplt.title(f\'AFM Scan: {channel_name}\')\nplt.xlabel(\'Pixels\')\nplt.ylabel(\'Pixels\')\n\n# Display the plot\nplt.show()\n\